# M04 — 工具與工具呼叫

本 notebook 對應同資料夾的 `README.md`，逐格執行即可。

目標：讓模型能呼叫外部能力。我們會定義 2 個工具，用 `bind_tools` 綁到模型，
讀懂 `ai.tool_calls`，然後**手動跑完一整輪工具迴圈**：
模型產生 tool_calls → 我們執行工具 → 用 `ToolMessage` 回填 → 再 invoke 拿最終答案。

## 1. 環境準備

跟前面模組一樣，透過 `_shared/course_utils.py` 取得供應商無關的模型。
這一格沒有可見輸出，成功代表 `model` 已就緒。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
model = get_model()

## 2. 定義兩個工具

用 `@tool` 把普通函式包裝成模型看得懂的工具。
注意三個給「模型」看的線索（不是給人看的裝飾）：

- 函式名稱 → 工具叫什麼
- docstring → 這個工具什麼時候該用
- type hint → 參數叫什麼、是什麼型別

缺了 docstring 或 type hint，模型就不知道何時用、怎麼填參數。
第二個工具 `get_weather` 回傳的是**假資料**，重點在示範「工具能取得模型不知道的外部資訊」。

In [ ]:
from langchain.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Add two numbers and return the sum."""
    return a + b


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    # Fake data for teaching; a real tool would call a weather API here.
    fake_db = {
        "台北": "晴, 31°C",
        "東京": "多雲, 24°C",
        "倫敦": "下雨, 15°C",
    }
    return fake_db.get(city, f"查無 {city} 的天氣資料")


# Inspect what the model will actually "see" for one tool.
print("工具名稱:", add.name)
print("工具描述:", add.description)
# Expected output (大致如下):
# 工具名稱: add
# 工具描述: Add two numbers and return the sum.

## 3. 用 `bind_tools` 把工具綁到模型

`bind_tools` 不改變原本的 model，而是回傳一個「知道有哪些工具可用」的新模型。
它仍然是個 Runnable，照樣用 `.invoke()`（這點和 M03 一致）。

In [ ]:
model_with_tools = model.bind_tools([add, get_weather])

## 4. 觀察 `ai.tool_calls` 的結構

對綁了工具的模型 invoke 一個需要算數的問題。
若模型決定用工具，回傳的 `AIMessage` 的 `.content` 通常是空字串，
真正的內容在 `.tool_calls`：一個 list，每筆含 name / args / id。

In [ ]:
ai = model_with_tools.invoke("3 加 5 是多少?")

print("content:", repr(ai.content))   # 多半是 ''，因為模型選擇用工具而非直接作答
print("tool_calls:", ai.tool_calls)
# Expected output (大致如下):
# content: ''
# tool_calls: [{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'call_abc...', 'type': 'tool_call'}]

## 5. 手動執行一輪工具迴圈

模型只是「說」它想呼叫 `add(a=3, b=5)`，並不會自己執行。
真正去跑函式、把結果送回去的是**我們的程式**。一輪流程：

1. 把使用者問題放進 messages，invoke 拿到帶 tool_calls 的 AIMessage
2. 依每筆 tool_call 找到對應工具、用它的 args 執行
3. 把結果包成 `ToolMessage`，`tool_call_id` 要對上 tool_call 的 id
4. 歷史順序：先放那個 AIMessage，再放 ToolMessage（成對出現）
5. 再 invoke 一次，模型看到工具結果後產出最終答案

用一個小迴圈處理 tool_calls，這樣模型一次要求呼叫多個工具也能應付。

In [ ]:
from langchain.messages import HumanMessage, ToolMessage

# Map tool name -> tool object so we can dispatch by name.
tools_by_name = {add.name: add, get_weather.name: get_weather}

# Step 1: first turn — model decides which tools to call.
messages = [HumanMessage("3 加 5 是多少? 另外台北現在天氣如何?")]
ai = model_with_tools.invoke(messages)
messages.append(ai)   # keep the AIMessage (with tool_calls) in history

print("這一輪模型要求呼叫的工具:")
for call in ai.tool_calls:
    print(" -", call["name"], call["args"])

# Step 2~3: run each requested tool and feed results back as ToolMessage.
for call in ai.tool_calls:
    selected_tool = tools_by_name[call["name"]]
    result = selected_tool.invoke(call["args"])   # actually execute the function
    messages.append(
        ToolMessage(content=str(result), tool_call_id=call["id"])
    )

# Step 5: second turn — model now sees the tool results and answers.
final = model_with_tools.invoke(messages)
print("\n最終答案:", final.content)
# Expected output (大致如下):
# 這一輪模型要求呼叫的工具:
#  - add {'a': 3, 'b': 5}
#  - get_weather {'city': '台北'}
#
# 最終答案: 3 加 5 等於 8。台北目前天氣為晴，31°C。

## 6. 把整輪流程包成一個函式

上面的步驟很常用，包成一個小函式方便重複呼叫。
注意這裡只跑「一輪」工具呼叫；真實任務可能要連續好幾輪，
那就需要一個 `while` 迴圈反覆判斷「還有 tool_calls 嗎?」——這正是 M06 要自動化的事。

In [ ]:
def run_one_tool_round(user_text: str) -> str:
    """Run a single tool-calling round and return the model's final answer."""
    messages = [HumanMessage(user_text)]
    ai = model_with_tools.invoke(messages)
    messages.append(ai)

    # No tool needed? Return the direct answer.
    if not ai.tool_calls:
        return ai.content

    # Execute every requested tool and append its ToolMessage.
    for call in ai.tool_calls:
        selected_tool = tools_by_name[call["name"]]
        result = selected_tool.invoke(call["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    # Ask the model again, now that it can see the tool outputs.
    return model_with_tools.invoke(messages).content


print(run_one_tool_round("倫敦現在天氣怎樣?"))
# Expected output (大致如下):
# 倫敦目前天氣為下雨，15°C。

## 🧪 練習 1：新增一個工具

加一個 `multiply(a: int, b: int) -> int` 工具，記得寫好 docstring 與 type hint。
把它一起 `bind_tools`，再問模型「6 乘以 7 是多少?」，
觀察 `ai.tool_calls` 是否正確選到 `multiply`、參數是否正確。

提示：別忘了把新工具也加進 `tools_by_name`，否則迴圈裡找不到它。

In [ ]:
# 在這裡寫你的答案
# @tool
# def multiply(a: int, b: int) -> int:
#     """Multiply two numbers and return the product."""
#     ...
#
# model_with_tools = model.bind_tools([add, get_weather, multiply])
# tools_by_name = {...}
# print(run_one_tool_round("6 乘以 7 是多少?"))

## 🧪 練習 2：故意拿掉 docstring 看會怎樣

複製 `get_weather`，做一個 `get_weather_bad`，但**把 docstring 刪掉**（或改成無意義的字）。
用它 `bind_tools` 後問「東京天氣?」，看看模型還能不能正確判斷要呼叫它、參數填得對不對。
體會 docstring 與 type hint 對「模型能不能用好工具」的影響。

In [ ]:
# 在這裡寫你的答案

## 小結 & 下一步

這一格回顧本模組：

- **工具 = 有說明書的函式**：`@tool` 從名稱、docstring、type hint 抽出模型需要的線索。
- **`bind_tools`** 讓模型知道有哪些工具可用；模型只「說」要呼叫什麼，放在 `ai.tool_calls`。
- **手動工具迴圈**：讀 tool_calls → 自己執行工具 → 用 `ToolMessage`（對上 `tool_call_id`）回填
  → 再 invoke 取得最終答案。回填時 AIMessage 與 ToolMessage 要成對放進歷史。

你剛剛手刻的這段「工具迴圈」，正是下一個 **M06 `create_agent`** 自動幫你做的事：
它會自己跑完「呼叫工具 → 回填 → 再判斷」這個迴圈（而且能連續跑很多輪），
直到模型產出最終答案。Agent 不是魔法，就是把你今天手動跑的迴圈自動化。

（下一站 M05 會先補上 Embedding 與 RAG，讓之後的 Agent 也能查外部知識。）